In [5]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
caminho_produto = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/produtos_clean.csv'
df_produto = pd.read_csv(caminho_produto)

caminho_vendas = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/vendas_clean.csv'
df_vendas = pd.read_csv(caminho_vendas)

In [7]:
df_matriz = df_vendas.groupby(['id_client', 'id_product'])['qtd'].sum().reset_index()
df_matriz['presenca'] = 1  # ignora quantidade — só 0 ou 1

matriz = df_matriz.pivot_table(
    index='id_client',
    columns='id_product',
    values='presenca',
    fill_value=0          # clientes que não compraram = 0
)

print(f'Clientes: {matriz.shape[0]} | Produtos: {matriz.shape[1]}')

Clientes: 49 | Produtos: 150


In [8]:
matriz_produto = matriz.T

sim = cosine_similarity(matriz_produto)
df_sim = pd.DataFrame(
    sim,
    index=matriz_produto.index,
    columns=matriz_produto.index
)

In [10]:
id_gps = df_produto[
    df_produto['name'] == 'GPS Garmin Vortex Maré Drift'
]['id_product'].values[0]

# Remove o próprio produto do ranking
similares = df_sim[id_gps].drop(id_gps).sort_values(ascending=False).head(5)

df_ranking = similares.reset_index()
df_ranking.columns = ['id_product', 'similaridade']
df_ranking = df_ranking.merge(
    df_produto[['id_product', 'name']],
    left_on='id_product',
    right_on='id_product'
)
df_ranking = df_ranking[['id_product', 'name', 'similaridade']]
df_ranking['similaridade'] = df_ranking['similaridade'].round(4)
df_ranking.index = range(1, 6)

print(df_ranking.to_string())

   id_product                                        name  similaridade
1          94            Motor de Popa Volvo Magnum 276HP        0.8696
2          11         GPS Furuno Swift Leviathan Poseidon        0.8680
3          35                          Radar Furuno Swift        0.8539
4           1                 Transponder AIS Maré Magnum        0.8500
5         115  Cabo de Nylon Delta Force Magnum Leviathan        0.8500
